# 파티 코치 — 재구성 데이터 전용 Colab
Qwen3.5-4B, 4bit QLoRA, 길이 4096, r=8. T4에서는 Qwen3.5의 float32 경로를 사용합니다.
샘플·파티 대화를 assistant 행동별로 학습하며 마지막 응답만 loss에 포함합니다.
학습과 채팅 모두 같은 compact_context 입력 표현을 사용합니다.
**처음에는 SMOKE_TEST=True로 가장 긴 데이터 2개만 확인합니다. 성공하면 런타임을 다시 시작하고 False로 바꿔 전체 실행하세요.**
기존 ZIP이나 기존 노트북과 혼용하지 마세요. 실제 T4 학습과 추천 품질은 실행 후 확인해야 합니다.


## 1. 드라이브 연결 + 설치 + 묶음 풀기
런타임마다 새로 설치한다 (몇 분 걸림).

In [ ]:
import os, sys, json, re, shutil, subprocess, zipfile, hashlib
from pathlib import Path
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
USE_DRIVE = True
DRIVE_DIR = Path('/content/drive/MyDrive/pokemon_coach')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'unsloth', 'gradio',
                'django>=5.2,<6', 'djangorestframework', 'jsonschema'], check=True)
BUNDLE_NAME = 'colab_bundle.zip'
paths = [Path('/content') / BUNDLE_NAME, DRIVE_DIR / BUNDLE_NAME]
def is_current_bundle(path):
    if not path.exists():
        return False
    with zipfile.ZipFile(path) as z:
        return 'llm/training/scripts/compact_context.py' in z.namelist() and 'llm/training/finetuning_data_ver2/report.json' in z.namelist()
bundle = next((p for p in sorted((p for p in paths if p.exists()), key=lambda p: p.stat().st_mtime, reverse=True)
               if is_current_bundle(p)), None)
if bundle is None:
    raise FileNotFoundError(f'{BUNDLE_NAME}을 Colab 파일 탭 또는 {DRIVE_DIR}에 올리세요.')
REPO = Path('/content/pokemon_rebuilt_20261007')
if REPO.exists():
    shutil.rmtree(REPO)
with zipfile.ZipFile(bundle) as z:
    z.extractall(REPO)
print('사용 묶음:', bundle)


## 2. 설정
SMOKE_TEST=True는 가장 긴 입력 2개로 메모리·정답 마스킹을 점검합니다.
성공하면 런타임을 다시 시작하고 False로 바꿔 전체를 실행하세요.
학습률·총 스텝·유형별 개수·길이를 출력합니다. 기존 결과나 체크포인트는 읽지 않습니다.


In [ ]:
import torch
SMOKE_TEST = True
MODEL_NAME = 'unsloth/Qwen3.5-4B'
LOAD_IN_4BIT, MAX_SEQ_LEN = True, 4096
LR, LR_SCHEDULER, WARMUP_RATIO = 1e-4, 'cosine', .05
EPOCHS, TRAIN_LIMIT = 1, None
BATCH, GRAD_ACC = 1, (1 if SMOKE_TEST else 8)
LORA_R, LORA_ALPHA = 8, 16
WEIGHT_DECAY, LOG_EVERY, EVAL_N, SEED = .01, 1, 12, 3407
if not torch.cuda.is_available():
    raise RuntimeError('Colab 런타임을 GPU로 변경하세요.')
BF16 = torch.cuda.get_device_capability()[0] >= 8
# Qwen3.5는 T4의 float16 경로를 사용할 수 없어 Unsloth가 float32로 전환한다.
FP16 = False
gpu = torch.cuda.get_device_properties(0)
VRAM_GB = gpu.total_memory / 2**30
DATA_DIR = REPO / 'llm/training/finetuning_data_ver2'
report = json.loads((DATA_DIR / 'report.json').read_text())
actual_hash = hashlib.sha256((REPO / 'data/pokemon.db').read_bytes()).hexdigest()
assert actual_hash == report['version']['db_sha256'], 'DB 버전 불일치'
OUT_DIR = (DRIVE_DIR if USE_DRIVE else Path('/content')) / 'outputs_rebuilt_20261007' / ('smoke' if SMOKE_TEST else 'full')
RESUME = False
def show_settings(**extra):
    settings = {'GPU': f'{gpu.name} ({VRAM_GB:.1f}GB)', '모델': MODEL_NAME,
                '정밀도': 'bf16' if BF16 else 'float32 (Qwen3.5/T4)', '길이': MAX_SEQ_LEN,
                '학습률': LR, '스케줄': LR_SCHEDULER, '에폭': EPOCHS,
                '배치': f'{BATCH} × {GRAD_ACC}', 'LoRA r / alpha': f'{LORA_R} / {LORA_ALPHA}',
                '2스텝 점검': SMOKE_TEST, **extra}
    print('\n'.join(f'{k}: {v}' for k, v in settings.items()))
show_settings()


## 3. 입력 표현·도구
전체 도구의 이름·필수 인자·타입을 간단히 표시합니다. 육성형 중복만 참조로 줄이고 원문은 유지합니다.
서버에 붙일 때도 이 입력 표현을 적용해야 합니다.


In [ ]:
for p in (REPO / 'llm', REPO / 'llm/training'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from agent.prompt import SYSTEM as ORIGINAL_SYSTEM, screen_context
from agent.tools import TOOLS as CLAUDE_TOOLS
from training.scripts.compact_context import system_prompt, prepare_messages, mask_last_response
SYSTEM = system_prompt(ORIGINAL_SYSTEM, CLAUDE_TOOLS)
assert hashlib.sha256(SYSTEM.encode()).hexdigest() == report['version']['system_sha256'], '입력 표현 버전 불일치'
print('전체 도구:', ', '.join(t['name'] for t in CLAUDE_TOOLS))


## 4. 재구성 데이터
train/eval은 마지막 assistant 행동별 자료입니다. 원래 전체 대화는 source_train/source_eval.jsonl에 보존했습니다.
원본 팀·육성형·변형을 연결해서 학습/평가를 분리했습니다. 자연어 설명은 사람 검수 대기입니다.


In [ ]:
from collections import Counter
def read(name):
    return [json.loads(line) for line in (DATA_DIR / name).read_text().splitlines() if line.strip()]
train_rows, eval_rows = read('train.jsonl'), read('eval.jsonl')
print('학습 행동:', len(train_rows), Counter(r['kind'] for r in train_rows))
print('평가 행동:', len(eval_rows), Counter(r['kind'] for r in eval_rows))
print('원본 분리:', report['split'])
assert not ({r['origin_component'] for r in train_rows} & {r['origin_component'] for r in eval_rows})
if SMOKE_TEST:
    train_rows = sorted(train_rows, key=lambda r: r['tokens'], reverse=True)[:2]
    print('가장 긴 입력 2개로 점검:', [(r['id'], r['tokens']) for r in train_rows])


## 5. 모델 불러오기 + LoRA
Qwen3.5는 이미지도 받는 모델이라 `FastModel`로 불러오고, 언어 부분에만 LoRA를 붙인다.

In [ ]:
import gc
for key in ('trainer', 'model', 'tokenizer', 'tok'):
    if key in globals():
        del globals()[key]
gc.collect()
torch.cuda.empty_cache()
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, load_in_4bit=LOAD_IN_4BIT, full_finetuning=False)
tok = getattr(tokenizer, 'tokenizer', tokenizer)     # processor면 안쪽 글자 tokenizer

model = FastModel.get_peft_model(
    model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=SEED)
model.print_trainable_parameters()
print(f'모델 올린 뒤 VRAM {torch.cuda.memory_reserved() / 2**30:.1f}GB')


## 6. 실제 토크나이저로 길이 재확인
4096 초과 자료가 있으면 중단합니다. 조용히 제외하거나 잘라서 학습하지 않습니다.


In [ ]:
import math
from datasets import Dataset
def render(messages, add_generation_prompt=False):
    return tok.apply_chat_template(messages, tokenize=False, tools=None,
                                   add_generation_prompt=add_generation_prompt, enable_thinking=False)
def to_dataset(rows, label):
    texts = [render(r['messages']) for r in rows]
    lens = [len(tok(t, add_special_tokens=False).input_ids) for t in texts]
    overflow = [(r['id'], n) for r, n in zip(rows, lens) if n > MAX_SEQ_LEN]
    if overflow:
        raise ValueError(f'{label} 길이 초과 (제외 없이 중단): {overflow[:8]}')
    print(f'{label}: {len(rows)}행, 최대 {max(lens)} / 평균 {sum(lens)//len(lens)}토큰, 초과 0')
    return Dataset.from_dict({'text': texts})
dataset = to_dataset(train_rows, '학습')
# 평가를 한 종류에서만 뽑지 않도록 종류별로 번갈아 뽑는다.
def balanced_rows(rows, limit):
    buckets = {k: [r for r in rows if r['kind'] == k] for k in sorted({r['kind'] for r in rows})}
    result = []
    while len(result) < limit and any(buckets.values()):
        for values in buckets.values():
            if values and len(result) < limit:
                result.append(values.pop(0))
    return result
eval_selected = balanced_rows(eval_rows, EVAL_N)
eval_dataset = to_dataset(eval_selected, '평가')
STEPS = 2 if SMOKE_TEST else math.ceil(len(dataset)/(BATCH*GRAD_ACC))*EPOCHS
WARMUP_STEPS = 0 if SMOKE_TEST else math.ceil(STEPS*WARMUP_RATIO)
EVAL_EVERY = max(STEPS//5, 1)
show_settings(**{'총 스텝': STEPS, 'warmup 스텝': WARMUP_STEPS, '평가 주기': EVAL_EVERY})
print('\n예시 정답:\n', dataset[0]['text'][-800:])


## 7. 학습
마지막 assistant 응답만 학습합니다. 입력·조회 결과·지난 답변은 -100으로 마스킹합니다.
2스텝 점검에서는 평가를 생략합니다. 본 학습에서는 loss·학습률·grad norm·VRAM·평가 loss를 표시합니다.
기존 체크포인트를 재개하지 않습니다. OOM이면 런타임을 다시 시작하세요.


In [ ]:
import time, inspect
from transformers import TrainerCallback, DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig
class LiveLog(TrainerCallback):
    """로그가 나올 때마다 한 줄로: 스텝, 에폭, loss, 평가 loss, 학습률, grad norm, VRAM, 남은 시간."""
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = self.step_end = time.time()
        self.start_step, self.eval_time, self.evals = state.global_step, 0.0, 0

    def on_step_end(self, args, state, control, **kw):
        self.step_end = time.time()

    def on_evaluate(self, args, state, control, **kw):       # 평가 시간은 따로 (남은 시간이 튀지 않게)
        self.eval_time += time.time() - self.step_end
        self.evals += 1

    def on_log(self, args, state, control, logs=None, **kw):
        if not logs or 'train_runtime' in logs:
            return
        s = state.global_step
        done = max(s - self.start_step, 1)
        per_step = (time.time() - self.t0 - self.eval_time) / done
        evals_left = (state.max_steps - s) // max(args.eval_steps or state.max_steps, 1)
        left = per_step * (state.max_steps - s) + evals_left * self.eval_time / max(self.evals, 1)
        parts = [f'step {s:>4}/{state.max_steps}', f'epoch {state.epoch:.2f}']
        if 'loss' in logs: parts.append(f'loss {logs["loss"]:.4f}')
        if 'eval_loss' in logs: parts.append(f'★ 평가 loss {logs["eval_loss"]:.4f}')
        if 'learning_rate' in logs: parts.append(f'lr {logs["learning_rate"]:.2e}')
        if 'grad_norm' in logs: parts.append(f'grad {logs["grad_norm"]:.2f}')
        parts += [f'VRAM {torch.cuda.max_memory_reserved() / 2**30:.1f}GB', f'남은 {left / 60:.0f}분']
        print(' | '.join(parts), flush=True)


config = dict(dataset_text_field='text', output_dir=str(OUT_DIR / 'checkpoints'),
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=1, gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS, max_steps=2 if SMOKE_TEST else -1,
    learning_rate=LR, lr_scheduler_type=LR_SCHEDULER, warmup_steps=WARMUP_STEPS,
    optim='adamw_8bit', weight_decay=WEIGHT_DECAY, bf16=BF16, fp16=FP16,
    bf16_full_eval=False, fp16_full_eval=False, packing=False,
    logging_steps=LOG_EVERY, eval_strategy='no' if SMOKE_TEST else 'steps', eval_steps=EVAL_EVERY,
    save_strategy='no' if SMOKE_TEST else 'steps', save_steps=EVAL_EVERY, save_total_limit=2,
    seed=SEED, report_to='none', disable_tqdm=True)
fields = getattr(SFTConfig, '__dataclass_fields__', {})
params = inspect.signature(SFTConfig.__init__).parameters
length_key = next((k for k in ('max_length', 'max_seq_length') if k in fields or k in params), None)
if length_key is None:
    raise RuntimeError('SFTConfig 길이 인자를 확인할 수 없음. 라이브러리 버전을 확인하세요.')
config[length_key] = MAX_SEQ_LEN
kwargs = dict(model=model, train_dataset=dataset, eval_dataset=eval_dataset,
              args=SFTConfig(**config), callbacks=[LiveLog()],
              data_collator=DataCollatorForSeq2Seq(tokenizer=tok, padding=True, label_pad_token_id=-100))
trainer_params = inspect.signature(SFTTrainer.__init__).parameters
kwargs['processing_class' if 'processing_class' in trainer_params else 'tokenizer'] = tok
trainer = SFTTrainer(**kwargs)
marker = tok.encode('<|im_start|>assistant\n', add_special_tokens=False)
for attr in ('train_dataset', 'eval_dataset'):
    ds = getattr(trainer, attr)
    ds = ds.map(lambda e: mask_last_response(e, marker), load_from_cache_file=False)
    if any(not any(x != -100 for x in r['labels']) for r in ds):
        raise ValueError('학습할 정답 토큰이 없는 자료')
    setattr(trainer, attr, ds)
print('정답 마스킹 확인:', sum(x != -100 for x in trainer.train_dataset[0]['labels']), '토큰')
stats = trainer.train(resume_from_checkpoint=None)
print(f"끝: {stats.metrics['train_runtime']/60:.1f}분, loss {stats.metrics['train_loss']:.4f}, "
      f"최대 VRAM {torch.cuda.max_memory_reserved()/2**30:.1f}GB")


### 7-1. 학습 곡선
왼쪽: train loss(학습, 선)·eval loss(평가, 점). Colab에 한글 글꼴이 없어 그래프 글자는 영어. 오른쪽: 학습률이 스텝마다 어떻게 바뀌었는지.

In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
series = lambda key: [(h['step'], h[key]) for h in hist if key in h]
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.8))
if series('loss'):
    a.plot(*zip(*series('loss')), label='train loss')
if series('eval_loss'):
    a.plot(*zip(*series('eval_loss')), 'o-', label='eval loss')
a.set_xlabel('step'); a.set_title('loss'); a.legend(); a.grid(alpha=.3)
if series('learning_rate'):
    b.plot(*zip(*series('learning_rate')), color='tab:orange')
b.set_xlabel('step'); b.set_title('learning rate'); b.grid(alpha=.3)
plt.tight_layout(); plt.show()
for h in hist:
    if 'eval_loss' in h:
        print(f"step {h['step']:>4}: 평가 loss {h['eval_loss']:.4f}")

## 8. 평가: 도구 인자까지 확인
평가 행동에 대해 도구 이름·필수 인자/타입·정확한 인자 일치를 비교합니다.
도구 없이 끝나는 답변은 출력과 정답을 사람이 비교합니다. 이 점수만으로 최종 추천 품질을 판단하지 않습니다.
2스텝 점검에서는 생략합니다.


In [ ]:
TOOL_CALL = re.compile(r'<tool_call>\s*<function=([\w-]+)>(.*?)</function>\s*</tool_call>', re.S)
PARAM = re.compile(r'<parameter=([\w-]+)>\n?(.*?)\n?</parameter>', re.S)

def parse_calls(text):
    """Qwen3.5 도구 호출 → [(이름, 인자)]. 값이 JSON이면 풀어서."""
    calls = []
    for name, body in TOOL_CALL.findall(text):
        args = {}
        for k, v in PARAM.findall(body):
            try:
                args[k] = json.loads(v)
            except json.JSONDecodeError:
                args[k] = v
        calls.append((name, args))
    return calls


from jsonschema import Draft202012Validator
validators = {t['name']: Draft202012Validator(t['input_schema']) for t in CLAUDE_TOOLS}
def generate(msgs, max_new_tokens=1024, prepared=False):
    messages = msgs if prepared else prepare_messages(msgs, SYSTEM)
    ids = tok(render(messages, add_generation_prompt=True), add_special_tokens=False, return_tensors='pt').to(model.device)
    room = MAX_SEQ_LEN - ids['input_ids'].shape[1]
    if room < 80:
        raise ValueError('대화 문맥이 가득 찼습니다. 새 대화에서 질문을 이어 주세요.')
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=min(max_new_tokens, room), do_sample=False, use_cache=True)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=False)
model.eval()
if not SMOKE_TEST:
    score = Counter()
    for r in balanced_rows(eval_rows, 24):
        gold = r['messages'][-1]
        output = generate(r['messages'][:-1], max_new_tokens=1024, prepared=True)
        calls = parse_calls(output)
        expected = [(c['function']['name'], c['function']['arguments']) for c in gold.get('tool_calls', [])]
        score['examples'] += 1
        if expected:
            score['tool_examples'] += 1
            score['name_match'] += [n for n, a in calls] == [n for n, a in expected]
            score['schema_valid'] += bool(calls) and all(n in validators and validators[n].is_valid(a) for n, a in calls)
            score['exact_args'] += calls == expected
        else:
            print(r['kind'], '\n예측:', output[:500], '\n정답:', gold['content'])
    print(dict(score))
else:
    print('2스텝 점검 모델의 품질 평가는 생략합니다.')


## 9. 저장
본 학습만 새 폴더에 어댑터를 저장합니다. 데이터·입력 표현·도구와 함께 보관해야 합니다.


In [ ]:
ADAPTER = OUT_DIR / 'qwen35-4b-coach-lora'
if not SMOKE_TEST:
    model.save_pretrained(ADAPTER)
    tokenizer.save_pretrained(ADAPTER)
    shutil.copy(DATA_DIR / 'report.json', ADAPTER / 'data_report.json')
    shutil.copy(REPO / 'llm/training/scripts/compact_context.py', ADAPTER / 'compact_context.py')
    print('저장:', ADAPTER)
else:
    print('2스텝 점검 결과는 어댑터로 저장하지 않습니다.')


In [ ]:
if not USE_DRIVE and not SMOKE_TEST:
    from google.colab import files
    shutil.make_archive(str(ADAPTER), 'zip', ADAPTER)
    files.download(str(ADAPTER) + '.zip')


## 10. 채팅방
실제 back/DB 도구를 실행합니다. 2스텝 점검 모델은 추천 품질을 판단할 수 없습니다.
학습과 동일한 입력 표현을 사용하고 도구 호출 인자를 먼저 검사합니다.
“더블 파티 하나 짜줘”, “더블 한카리아스 샘플 짜줘”를 먼저 비교해 보세요.


In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
import gradio as gr
from inproc_back import back_client          # back(Django) + DB를 이 프로세스에서 (서버 안 띄움)
from agent.tools import Tools
from apps.api.common import names
from apps.dex.models import Ruleset

MAX_ROUNDS = 6                                # 한 번 답할 때 도구를 주고받는 최대 횟수 (서버와 같게)
RID = Ruleset.objects.order_by('-start_date').first().id
CHAT = {'turns': [], 'party': []}            # 지난 대화(사용자, 최종 답변) · 현재 파티

def ko(kind, key):
    row = names(RID)[kind].get(key) or {}
    return row.get('name_ko') or key

def readable(s):
    sp = ' '.join(f'{k}{v}' for k, v in (s.get('sp') or {}).items() if v)
    return (f"{ko('pokemon', s['pokemon'])} @ {ko('item', s.get('item', '')) or '-'} / {ko('ability', s.get('ability', ''))} / "
            f"{ko('nature', s.get('nature', ''))} / SP {sp} / " + ', '.join(ko('move', m) for m in s.get('moves', []) if m))

def clean(text):
    for pat in (r'<think>.*?</think>', r'<tool_call>.*?</tool_call>', r'<\|im_end\|>.*', r'<\|endoftext\|>'):
        text = re.sub(pat, '', text, flags=re.S)
    return text.strip()

async def agent_turn(question, fmt):
    """서버의 build_messages + run_agent와 같은 흐름: 지난 대화는 글만, 마지막 질문 앞에 [화면] 상황."""
    msgs = [{'role': 'system', 'content': SYSTEM}]
    for q, a in CHAT['turns']:
        msgs += [{'role': 'user', 'content': q}, {'role': 'assistant', 'content': a}]
    party = CHAT['party'] + [None] * (6 - len(CHAT['party']))
    msgs.append({'role': 'user', 'content': screen_context(fmt, party) + '\n\n' + question})
    log = []
    async with back_client() as c:
        tools = Tools(c, fmt)
        for _ in range(MAX_ROUNDS):
            try:
                text = generate(msgs, prepared=False)
            except ValueError as e:
                return str(e), log
            calls, reply = parse_calls(text), clean(text)
            if not calls:
                return reply, log
            msgs.append({'role': 'assistant', 'content': reply, 'tool_calls': [
                {'type': 'function', 'function': {'name': n, 'arguments': a}} for n, a in calls]})
            for n, a in calls:
                try:
                    if n not in validators:
                        raise ValueError(f'없는 도구: {n}')
                    validators[n].validate(a)
                    out = json.dumps(await tools.run(n, a), ensure_ascii=False, separators=(',', ':'))
                except Exception as e:                 # 없는 ID, 잘못된 입력 → 모델에게 오류로 돌려줌 (서버와 같게)
                    out = f'오류: {e}'
                msgs.append({'role': 'tool', 'content': out})
                log.append(f'🔧 {n}({json.dumps(a, ensure_ascii=False)[:120]}) → {out[:100]}')
                if n == 'propose_party' and tools.proposed:
                    CHAT['party'] = tools.proposed
                    tools.proposed = None
                    log.append('📋 추천 파티를 현재 파티로:\n' + '\n'.join(f'  {i}. {readable(s)}' for i, s in enumerate(CHAT['party'], 1)))
    return '(도구 호출이 너무 많아 멈췄어요)', log

def respond(message, history, fmt):
    if not history:                              # 새 대화 (지우기 포함)
        CHAT['turns'], CHAT['party'] = [], []
    fmt = 'doubles' if fmt == '더블' else 'singles'
    t0 = time.time()
    reply, log = ThreadPoolExecutor(1).submit(asyncio.run, agent_turn(message, fmt)).result()
    CHAT['turns'].append((message, reply))
    tail = f'\n\n---\n' + '\n'.join(log) if log else ''
    return f'{reply}{tail}\n\n_({time.time() - t0:.0f}초)_'

fmt_box = gr.Radio(['더블', '싱글'], value='더블', label='포맷')
try:
    demo = gr.ChatInterface(respond, additional_inputs=[fmt_box], type='messages', title='파티 코치 (파인튜닝 모델)',
                            examples=[['한카리아스 더블 샘플 짜줘', '더블'], ['요즘 싱글 픽률 1위 뭐야?', '싱글']])
except TypeError:                                # type 인자가 없는 gradio 버전
    demo = gr.ChatInterface(respond, additional_inputs=[fmt_box], title='파티 코치 (파인튜닝 모델)')
demo.launch(debug=False)


전체 대화·원본 ID·제외 사유는 ZIP의 source_train.jsonl/source_eval.jsonl/report.json에 있습니다.
자동 검사는 설명의 정확성이나 실제 추천 성능을 보장하지 않습니다. 마지막 채팅에서 조건 변경과 잘못된 이름도 확인하세요.
